In [ ]:
# ── Cài Đặt Thư Viện Đọc PLY & Thư Viện Biểu Diễn 3D Trực Quan Plotly ─────────
!pip install -q plyfile plotly numpy pandas

In [ ]:
# ── KHÂU 1: Hiển Thị Mây Điểm Ban Đầu (Iter 0) Tương Tác 3D Trực Quan ───────────────
import os
import numpy as np
from plyfile import PlyData
import plotly.graph_objects as go

def display_interactive_point_cloud_plotly(ply_path, max_points=50000):
    """
    Hiển thị Khung Mây Điểm 3D Tương Tác Trực Quan (Native VS Code, Google Colab & Kaggle).
    - Hỗ trợ Xoay 360 độ (Trái/Phải/Lên/Xuống), Phóng to/Thu nhỏ (Scroll wheel).
    - Nút '▶ Tự Động Quay 360°' và '⏸ Tạm Dừng' tự động xoay theo quỹ đạo camera.
    - Hiển thị đầy đủ màu sắc RGB chuẩn xác từ mây điểm 3DGS / COLMAP.
    """
    if not os.path.exists(ply_path):
        print(f"❌ Không tìm thấy file point cloud tại: {ply_path}")
        return

    print(f"📂 Đang nạp Mây Điểm 3D Khâu 1: {ply_path} ...")
    plydata = PlyData.read(ply_path)
    vertex = plydata['vertex']

    x = np.asarray(vertex['x'], dtype=np.float32)
    y = np.asarray(vertex['y'], dtype=np.float32)
    z = np.asarray(vertex['z'], dtype=np.float32)

    # Giải mã màu sắc từ các thuộc tính PLY
    names = [p.name for p in vertex.properties]
    if 'red' in names and 'green' in names and 'blue' in names:
        r = np.asarray(vertex['red'], dtype=np.float32)
        g = np.asarray(vertex['green'], dtype=np.float32)
        b = np.asarray(vertex['blue'], dtype=np.float32)
        if r.max() > 1.0:
            r, g, b = r / 255.0, g / 255.0, b / 255.0
    elif 'f_dc_0' in names and 'f_dc_1' in names and 'f_dc_2' in names:
        f_dc_0 = np.asarray(vertex['f_dc_0'], dtype=np.float32)
        f_dc_1 = np.asarray(vertex['f_dc_1'], dtype=np.float32)
        f_dc_2 = np.asarray(vertex['f_dc_2'], dtype=np.float32)
        SH_C0 = 0.28209479177387814
        r = np.clip(0.5 + SH_C0 * f_dc_0, 0.0, 1.0)
        g = np.clip(0.5 + SH_C0 * f_dc_1, 0.0, 1.0)
        b = np.clip(0.5 + SH_C0 * f_dc_2, 0.0, 1.0)
    else:
        r = g = b = np.ones_like(x) * 0.8

    total_points = len(x)
    if total_points > max_points:
        idx = np.random.choice(total_points, max_points, replace=False)
        x, y, z = x[idx], y[idx], z[idx]
        r, g, b = r[idx], g[idx], b[idx]

    # Chuẩn hóa màu hex RGB cho Plotly
    colors_hex = [f'rgb({int(ri*255)},{int(gi*255)},{int(bi*255)})' for ri, gi, bi in zip(r, g, b)]

    # Tạo đối tượng biểu diễn Mây Điểm 3D Scatter
    scatter = go.Scatter3d(
        x=x, y=y, z=z,
        mode='markers',
        marker=dict(
            size=2.2,
            color=colors_hex,
            opacity=0.92
        ),
        hoverinfo='none'
    )

    # Khởi tạo giao diện đồ họa 3D đẳng cấp
    fig = go.Figure(data=[scatter])

    # Tạo 60 khung hình quay tròn 360 độ tự động (Turntable Rotation)
    frames = []
    n_frames = 60
    r_cam = 2.2
    for i in range(n_frames):
        theta = 2 * np.pi * i / n_frames
        cam_x = r_cam * np.cos(theta)
        cam_y = r_cam * np.sin(theta)
        frames.append(go.Frame(
            layout=dict(
                scene_camera=dict(
                    eye=dict(x=cam_x, y=cam_y, z=0.8),
                    center=dict(x=0, y=0, z=0),
                    up=dict(x=0, y=0, z=1)
                )
            ),
            name=f"frame_{i}"
        ))
    fig.frames = frames

    # Thiết lập giao diện điều khiển với nút Tự Động Quay 360° & Tạm Dừng
    fig.update_layout(
        title=dict(
            text=f"🌐 <b>Khâu 1: Mây Điểm Ban Đầu (Iter 0)</b> | Số điểm: <b>{total_points:,}</b>",
            x=0.02, y=0.97,
            font=dict(size=16, color='#60a5fa')
        ),
        scene=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            aspectmode='data',
            bgcolor='#0f172a'
        ),
        paper_bgcolor='#0f172a',
        font=dict(color='#e2e8f0', family='system-ui'),
        margin=dict(l=0, r=0, b=0, t=40),
        height=680,
        updatemenus=[
            dict(
                type="buttons",
                showactive=True,
                y=0.95, x=0.02,
                xanchor="left", yanchor="top",
                pad=dict(t=0, r=10),
                bgcolor='rgba(30, 41, 59, 0.85)',
                bordercolor='rgba(255, 255, 255, 0.2)',
                font=dict(color='#ffffff', size=13),
                buttons=[
                    dict(
                        label="▶ Tự Động Quay 360°",
                        method="animate",
                        args=[None, dict(
                            frame=dict(duration=50, redraw=False),
                            fromcurrent=True,
                            mode="immediate",
                            loop=True
                        )]
                    ),
                    dict(
                        label="⏸ Tạm Dừng (Tự Tương Tác Chuột)",
                        method="animate",
                        args=[[None], dict(
                            frame=dict(duration=0, redraw=False),
                            mode="immediate",
                            transition=dict(duration=0)
                        )]
                    )
                ]
            )
        ]
    )

    print("✨ Đã dựng thành công Khung Mây Điểm 3D Tương Tác!")
    fig.show()

# ── Đọc và Hiển Thị File Mây Điểm Local ─────────────────────────────────────────
local_base = os.path.join(os.getcwd(), "counter_demo_video")
ply_path = os.path.join(local_base, "step1_initial_points", "initial_point_cloud.ply")

if not os.path.exists(ply_path):
    ply_path = r"C:\Users\YUT9HC\Desktop\Z\thesis-all\counter_demo_video\step1_initial_points\initial_point_cloud.ply"

if os.path.exists(ply_path):
    display_interactive_point_cloud_plotly(ply_path)
else:
    print(f"⚠️ Không tìm thấy file tại {ply_path}. Vui lòng kiểm tra lại thư mục counter_demo_video!")